# 第6章 智能视觉系统开发

本节是第 6 章的导引课，将带你建立智能视觉系统开发的完整认知框架，理解在昇腾 NPU 上开发视觉应用的全流程与关键技术。

你将学到：
1. 智能视觉系统的基本概念与组成
2. 视觉系统开发的完整流程
3. 昇腾视觉开发的三大关键技术：DVPP、AIPP、AscendCL
4. 模型部署链路：PT → ONNX → OM
5. 在昇腾 NPU 上亲手运行视觉推理代码

> **运行环境**：CANN 9.0.0 · Python 3.11 · 昇腾 910B3 NPU · 16vCPUs · 32GiB

---

## 1. 什么是智能视觉系统

**智能视觉系统**是让计算机"看懂"图像和视频，并做出决策的系统。它是人工智能最重要的应用领域之一，涵盖目标检测、人脸识别、图像分类、视频分析等任务。

<img src="../../images/vision_architecture.png" alt="昇腾智能视觉系统架构" style="display: block; margin-left: 0;" />

一个完整的智能视觉系统从下到上分为五层：

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">层次</th><th style="text-align: left;">职责</th><th style="text-align: left;">关键技术</th></tr>
<tr><td style="text-align: left;">数据层</td><td style="text-align: left;">采集图像/视频流，送入设备内存</td><td style="text-align: left;">摄像头、JPEG/YUV 编码</td></tr>
<tr><td style="text-align: left;">硬件层</td><td style="text-align: left;">执行 AI 计算与视频预处理</td><td style="text-align: left;">昇腾 NPU（AI Core + DVPP 单元）</td></tr>
<tr><td style="text-align: left;">CANN 层</td><td style="text-align: left;">模型转换、图优化、算子库、硬件预处理</td><td style="text-align: left;">ATC、GE、AIPP、DVPP</td></tr>
<tr><td style="text-align: left;">框架层</td><td style="text-align: left;">提供开发 API，管理计算图</td><td style="text-align: left;">PyTorch、MindSpore、AscendCL</td></tr>
<tr><td style="text-align: left;">应用层</td><td style="text-align: left;">面向用户的视觉应用</td><td style="text-align: left;">检测、识别、分类、分析</td></tr>
</table>

上表从下到上描述了智能视觉系统的五层架构。**数据层**负责图像采集，是整个系统的输入端，常见数据源包括 USB 摄像头、网络摄像头（RTSP 视频流）、本地图片文件等。**硬件层**是系统的计算核心，昇腾 NPU 内部包含 AI Core（负责神经网络推理）和 DVPP 单元（负责图像编解码与几何变换），两者共享同一颗芯片。**CANN 层**是软件栈的核心，ATC 负责模型转换与图优化，AIPP 负责硬件预处理配置，DVPP 提供硬件解码 API。**框架层**提供开发者使用的 API，PyTorch 和 MindSpore 用于训练，AscendCL 用于部署推理。**应用层**是最终面向用户的视觉功能，如人脸门禁、缺陷检测、车牌识别等。

> 类比：智能视觉系统就像一条"视觉流水线"——摄像头是原料入口（数据层），NPU 是加工车间（硬件层），CANN 是车间调度系统（CANN层），框架是操作手册（框架层），最终产品是检测结果（应用层）。

## 2. 视觉系统开发全流程

在昇腾平台上开发一个智能视觉系统，通常经历以下六个阶段：

<img src="../../images/vision_pipeline.png" alt="视觉系统开发全流程" style="display: block; margin-left: 0;" />

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">阶段</th><th style="text-align: left;">做什么</th><th style="text-align: left;">昇腾工具</th></tr>
<tr><td style="text-align: left;">① 数据采集</td><td style="text-align: left;">收集图像/视频数据，标注</td><td style="text-align: left;">—</td></tr>
<tr><td style="text-align: left;">② 图像预处理</td><td style="text-align: left;">解码、缩放、归一化、通道转换</td><td style="text-align: left;">DVPP（硬件）/ AIPP（硬件）</td></tr>
<tr><td style="text-align: left;">③ 模型训练</td><td style="text-align: left;">用标注数据训练模型权重</td><td style="text-align: left;">PyTorch + torch_npu</td></tr>
<tr><td style="text-align: left;">④ 模型转换</td><td style="text-align: left;">将 .pt/.onnx 转为 NPU 专属 .om</td><td style="text-align: left;">ATC（模型转换工具）</td></tr>
<tr><td style="text-align: left;">⑤ 部署推理</td><td style="text-align: left;">加载 .om 模型，在 NPU 上推理</td><td style="text-align: left;">AscendCL（C++/Python API）</td></tr>
<tr><td style="text-align: left;">⑥ 后处理输出</td><td style="text-align: left;">解码预测结果，绘制框/关键点</td><td style="text-align: left;">Python/C++</td></tr>
</table>

上表列出了视觉系统开发的六个阶段。阶段①数据采集和阶段③模型训练与通用深度学习流程相同，不在本课程重点讨论。阶段②图像预处理是昇腾平台的特色——用 DVPP 硬件替代 OpenCV 软件做解码和缩放，用 AIPP 硬件替代 CPU 做归一化和通道转换。阶段④模型转换通过 ATC 工具将通用的 ONNX 模型编译为昇腾专属的 OM 离线模型，期间自动进行图融合优化。阶段⑤部署推理使用 AscendCL API 加载 OM 模型并在 NPU 上执行推理。阶段⑥后处理输出将模型预测的坐标和置信度解码为最终的检测框、关键点等可视化结果。

> **关键洞察**：阶段②和④是昇腾区别于 GPU 平台的核心——DVPP 和 AIPP 把预处理从 CPU 卸载到 NPU 专用硬件，ATC 把通用模型编译成 NPU 最优指令，这是昇腾视觉系统高性能的来源。

## 3. 昇腾视觉开发三大关键技术

### 3.1 DVPP —— 硬件视频预处理

**DVPP**（Digital Vision Pre-Processing，数字视频预处理）是昇腾 NPU 内置的专用视频处理单元，能用硬件加速完成 JPEG 解码、图像缩放、色彩转换等操作。

<img src="../../images/dvpp_flow.png" alt="DVPP流水线" style="display: block; margin-left: 0;" />

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">功能</th><th style="text-align: left;">CPU 实现</th><th style="text-align: left;">DVPP 硬件</th></tr>
<tr><td style="text-align: left;">JPEG 解码</td><td style="text-align: left;">OpenCV imdecode（软件）</td><td style="text-align: left;">dvpp_jpeg_decode_async（硬件）</td></tr>
<tr><td style="text-align: left;">图像缩放</td><td style="text-align: left;">OpenCV resize（软件）</td><td style="text-align: left;">dvpp_vpc_resize_async（硬件）</td></tr>
<tr><td style="text-align: left;">色彩转换</td><td style="text-align: left;">OpenCV cvtColor（软件）</td><td style="text-align: left;">AIPP 硬件或 DVPP（硬件）</td></tr>
</table>

上表列出了三种最常见的图像预处理操作。以 JPEG 解码为例，OpenCV 的 `imdecode` 需要在 CPU 上逐字节解析 JPEG 压缩流、做哈夫曼解码和 IDCT 反变换，耗时通常在 2~5 ms；而 DVPP 的 `dvpp_jpeg_decode_async` 调用 NPU 内部的硬解码器，只需 0.3~0.8 ms 即可完成同样的解码工作，且解码后的 YUV 数据直接留在 Device 内存中，无需再拷贝回 Host。图像缩放和色彩转换也有类似的量级差异。

> 优势：DVPP 与 AI Core 共享 NPU 芯片，预处理和推理在片内流水衔接，无需跨设备数据搬运，大幅降低延迟。

### 3.1.1 OpenCV 链路与 DVPP 链路的对比

在实际视觉系统开发中，图像预处理可以选择 **OpenCV 软件链路** 或 **DVPP 硬件链路**，两者各有特点：

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">对比维度</th><th style="text-align: left;">OpenCV 链路（CPU 软件）</th><th style="text-align: left;">DVPP 链路（NPU 硬件）</th></tr>
<tr><td style="text-align: left;">执行硬件</td><td style="text-align: left;">CPU（通用处理器）</td><td style="text-align: left;">NPU 内置 DVPP 专用单元</td></tr>
<tr><td style="text-align: left;">数据位置</td><td style="text-align: left;">Host 内存，需 H2D 拷贝到 Device</td><td style="text-align: left;">直接在 Device 内存完成，无需拷贝</td></tr>
<tr><td style="text-align: left;">处理速度</td><td style="text-align: left;">较慢（受 CPU 主频和带宽限制）</td><td style="text-align: left;">快 3~10 倍（专用硬件流水线）</td></tr>
<tr><td style="text-align: left;">格式灵活性</td><td style="text-align: left;">极高（支持任意格式、任意尺寸）</td><td style="text-align: left;">有限（需 128/16 字节对齐，格式受限）</td></tr>
<tr><td style="text-align: left;">CPU 占用</td><td style="text-align: left;">高（占用 CPU 核心做解码/缩放）</td><td style="text-align: left;">低（CPU 只需下发指令，硬件异步执行）</td></tr>
<tr><td style="text-align: left;">开发难度</td><td style="text-align: left;">低（API 简单，资料丰富）</td><td style="text-align: left;">中（需理解对齐规则和异步编程）</td></tr>
</table>

**各自适用场景**：

- **OpenCV 链路** 适用于：① 原型开发和快速验证（无需关心对齐）；② 非标准格式图像处理（如 BMP、TIFF、16 位图）；③ CPU 算力充足且推理频率不高的场景；④ 需要复杂图像处理算法（如仿射变换、滤波、轮廓检测）超出 DVPP 能力范围的场景。

- **DVPP 链路** 适用于：① 高吞吐量生产部署（如视频流 25/30 fps 实时检测）；② 端侧设备（CPU 算力有限，需把 CPU 留给后处理和业务逻辑）；③ 低延迟要求场景（如自动驾驶、工业质检）；④ 大批量图像流水线处理（如批量人脸识别门禁系统）。

> **实践建议**：在开发阶段先用 OpenCV 链路验证算法正确性，部署阶段切换到 DVPP 链路追求极致性能。两者并非互斥——实际系统中常混合使用：OpenCV 做复杂预处理（如 letterbox 填充），DVPP 做标准解码和缩放，AIPP 做归一化，三者协同构成完整的视觉预处理流水线。

### 3.1.2 DVPP 功能详解

📹 **DVPP：高效的图片/视频硬件加速器**

DVPP（Digital Video Pre-Processing）是昇腾 AI 处理器内置的专用图像处理单元。你可以把它看作一个独立的硬件模块，专门负责高强度的媒体数据处理，从而将 AI Core（负责推理的核心）从这些繁杂的任务中解放出来，实现真正的硬件加速。

它就像一个高效的"数据预处理工厂"，主要提供以下硬件加速功能：

**图片处理（VPC - Vision Preprocessing Core）：**

- 图像缩放（Resize）：支持华为自研高阶滤波算法和业界 Bilinear 算法。
- 图像抠图（Crop）：支持从图片中抠出指定区域。
- 格式转换（CSC）：支持 YUV 和 RGB 格式之间的转换。
- 图像金字塔等高级处理功能。

**图片编解码：**

- JPEGD：将 JPEG 格式图片解码为 YUV 格式。
- JPEGE：将 YUV 格式图片编码为 JPEG 格式。
- PNGD：将 PNG 格式图片解码为 RGB 格式。

**视频编解码：**

- VDEC：将 H.264/H.265 视频码流解码为 YUV/RGB 格式。
- VENC：将 YUV420SP 格式视频编码为 H.264/H.265 格式。

> **注意**：DVPP 在处理数据时存在一些硬件约束，例如输出格式可能受限（如仅支持 YUV），或对图片的宽/高有对齐要求。

### 3.2 AIPP —— 硬件 AI 预处理

**AIPP**（AI Pre-Processing，人工智能预处理）是昇腾在模型推理流水线上内置的预处理单元。它把每次推理都要重复做的预处理计算（归一化、色域转换、通道重排等）从 CPU 代码中"抽"出来，在 ATC 转换时静态写进 OM 模型，由 NPU 在数据进入网络第一层之前用专用硬件完成。

<img src="../../images/aipp_compare.png" alt="AIPP对比" style="display: block; margin-left: 0;" />

AIPP 的三大收益：

1. **计算卸载**：归一化 `/255` 与通道转置 `HWC→CHW` 从 CPU 移到 NPU 专用硬件，CPU 预处理时间归零
2. **带宽节省**：Host→Device 传输量从 float32（4 字节）降为 uint8（1 字节），拷贝量约为原来的 1/4
3. **流水衔接**：预处理与网络第一层在 NPU 内部无缝衔接，无跨设备同步开销

AIPP 配置文件（`aipp.cfg`）核心内容：

```protobuf
aipp_op {
  aipp_mode: static          # 静态 AIPP，转换时固化进 OM
  input_format: RGB888_U8    # 输入为 RGB uint8
  csc_switch: false          # 关闭色域转换（Python 已完成 BGR→RGB）
  # 归一化: y = (x - min_chn) * var_reci_chn，即 y = x / 255
  min_chn_0: 0.0
  var_reci_chn_0: 0.00392157   # = 1/255
}
```

### 3.2.1 AIPP 两种模式：静态 vs 动态

🧠 **AIPP：灵活的模型输入"适配器"**

AIPP（Artificial Intelligence Pre-Processing）是在 AI Core 上完成数据预处理的一种机制。与 DVPP 不同，AIPP 并非一个独立的硬件单元，更像一个嵌入在模型中的预处理指令集。

AIPP 提供了两种模式，灵活性大不相同：

**静态 AIPP：**

- **操作方式**：在模型转换（ATC）时通过配置文件（`.cfg` 或 `.aippconfig`）设置参数，这些参数会固化在最终的 `.om` 模型文件中。
- **灵活性**：低。参数在模型推理期间固定不变，无法修改。
- **适用场景**：输入数据格式和预处理要求完全固定的场景。

**动态 AIPP：**

- **操作方式**：模型转换时仅开启动态模式，不固化具体参数。在运行时，通过代码调用专门的 API（如 `aclmdlSetInputAIPP`）来设置预处理参数。
- **灵活性**：高。你可以在每次推理前，根据实际需求动态调整参数。
- **适用场景**：需要处理不同来源（如不同摄像头）、不同格式或不同预处理参数的数据。甚至可以在多 batch（批量）推理中，为每个 batch 设置不同的 AIPP 参数。

### 3.3 AscendCL —— 推理运行时框架

**AscendCL**（Ascend Computing Language）是昇腾提供的 C++/Python API，用于管理 NPU 资源、加载模型、执行推理。它是连接应用代码与 NPU 硬件的直接接口。

<img src="../../images/acl_lifecycle.png" alt="AscendCL生命周期" style="display: block; margin-left: 0;" />

AscendCL 的核心模块：

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">模块</th><th style="text-align: left;">功能</th><th style="text-align: left;">关键 API</th></tr>
<tr><td style="text-align: left;">运行时管理</td><td style="text-align: left;">设备/上下文/流管理</td><td style="text-align: left;">acl.rt.set_device / create_context</td></tr>
<tr><td style="text-align: left;">模型管理</td><td style="text-align: left;">加载/执行 OM 模型</td><td style="text-align: left;">acl.mdl.load_from_file / execute</td></tr>
<tr><td style="text-align: left;">内存管理</td><td style="text-align: left;">Device 内存分配/拷贝</td><td style="text-align: left;">acl.rt.malloc / memcpy</td></tr>
<tr><td style="text-align: left;">媒体数据处理</td><td style="text-align: left;">DVPP 视频预处理</td><td style="text-align: left;">acl.media.dvpp_*</td></tr>
</table>

上表列出了 AscendCL 的四个核心模块。**运行时管理**模块负责初始化 NPU 设备、创建执行上下文和流（Stream），是所有 AscendCL 程序的第一步。**模型管理**模块负责加载 OM 模型文件、获取输入输出描述信息并执行推理，`acl.mdl.load_from_file` 将 OM 文件加载到 Device 内存，`acl.mdl.execute` 触发一次同步推理。**内存管理**模块负责在 Device 上分配和释放内存，以及 Host 与 Device 之间的数据拷贝。**媒体数据处理**模块封装了 DVPP 的全部能力，包括 JPEG 解码/编码、VPC 缩放/裁剪、色彩转换等，API 命名统一以 `acl.media.dvpp_` 开头。

AscendCL 的典型调用流程为：`acl.init()` → `acl.rt.set_device(0)` → `acl.rt.create_context()` → `acl.mdl.load_from_file()` → `acl.rt.malloc()` 分配输入输出内存 → `acl.rt.memcpy()` 拷贝输入到 Device → `acl.mdl.execute()` 推理 → `acl.rt.memcpy()` 拷贝输出回 Host → 释放所有资源 → `acl.finalize()`。每一步都必须显式管理，这是 C 语言风格 API 的特点，虽然代码量比 PyTorch 多，但能精确控制资源，适合生产部署。

### 3.4 DVPP + AIPP 最佳实践组合

🤝 **DVPP + AIPP：最佳实践组合**

在实际开发中，DVPP 和 AIPP 常常组合使用，各司其职，形成一条高效的数据预处理流水线。一个典型的流程是：

1. **DVPP 先行**：首先使用 DVPP 对原始的图片或视频进行硬加速，完成解码、缩放、抠图等通用、计算量大的任务。
2. **AIPP 微调**：由于 DVPP 的输出可能（因硬件约束）不完全符合模型输入要求，再将 DVPP 的输出送入 AIPP，进行最后的色域转换、精确抠图/填充、减均值/乘系数（归一化）等操作。
3. **模型推理**：AIPP 处理完毕的数据直接满足模型输入要求，随后进行推理。

📊 **核心差异对比**

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">特性</th><th style="text-align: left;">DVPP</th><th style="text-align: left;">AIPP</th></tr>
<tr><td style="text-align: left;">本质</td><td style="text-align: left;">专用的硬件处理单元</td><td style="text-align: left;">AI Core 上的预处理机制</td></tr>
<tr><td style="text-align: left;">主要功能</td><td style="text-align: left;">图片/视频的编解码、缩放、抠图等</td><td style="text-align: left;">色域转换、抠图/填充、减均值/乘系数（归一化）</td></tr>
<tr><td style="text-align: left;">处理位置</td><td style="text-align: left;">独立的图像处理单元</td><td style="text-align: left;">AI Core</td></tr>
<tr><td style="text-align: left;">灵活性</td><td style="text-align: left;">通过代码调用 API，编程灵活</td><td style="text-align: left;">静态 AIPP（固化，不灵活）vs 动态 AIPP（运行时设置，灵活）</td></tr>
<tr><td style="text-align: left;">使用方式</td><td style="text-align: left;">通过 AscendCL 接口在代码中调用</td><td style="text-align: left;">在模型转换时配置，或在推理代码中动态设置</td></tr>
</table>

💎 **总结**

- **DVPP** 是硬件加速器，专注于"重体力活"（编解码、缩放），以固定功能 API 的形式提供高性能处理。
- **AIPP** 是模型输入"适配器"，专注于"精细调整"（色域、归一化），通过静态或动态配置提供灵活性，使模型输入更加规范。

理解它们的定位和组合使用方式，是充分发挥昇腾硬件性能的关键。

## 4. 模型部署链路：PT → ONNX → OM

在昇腾平台上，模型从训练到部署经历一条标准转换链路：

<img src="../../images/deploy_pipeline.png" alt="模型部署链路" style="display: block; margin-left: 0;" />

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">格式</th><th style="text-align: left;">说明</th><th style="text-align: left;">转换工具</th></tr>
<tr><td style="text-align: left;">.pt</td><td style="text-align: left;">PyTorch 权重文件，训练产物</td><td style="text-align: left;">PyTorch 训练</td></tr>
<tr><td style="text-align: left;">.onnx</td><td style="text-align: left;">开放神经网络交换格式，跨平台</td><td style="text-align: left;">torch.onnx.export</td></tr>
<tr><td style="text-align: left;">.om</td><td style="text-align: left;">昇腾离线模型，NPU 最优指令</td><td style="text-align: left;">ATC（含图融合优化）</td></tr>
<tr><td style="text-align: left;">-aipp.om</td><td style="text-align: left;">带 AIPP 预处理的 OM，"自带预处理"</td><td style="text-align: left;">ATC + --insert_op_conf</td></tr>
</table>

上表展示了模型从训练到部署的四种格式。`.pt` 文件是 PyTorch 训练产物，包含模型结构和权重，只能在 PyTorch 框架中使用。`.onnx` 是开放神经网络交换格式，使用 `torch.onnx.export` 导出，可跨平台运行（CPU、GPU、NPU），是模型交付的通用中间格式。`.om` 是昇腾离线模型，通过 ATC 工具从 ONNX 编译而来，内部是 NPU 专用的最优指令序列，只能在昇腾 NPU 上运行，但推理速度远超 ONNX Runtime。`-aipp.om` 在 OM 基础上额外固化了 AIPP 预处理配置，模型"自带预处理"，输入直接传 uint8 图像数据即可，无需在 Python 中做归一化和通道转换。

**ATC 命令示例**：

```bash
# 纯 OM 转换
atc --model=model.onnx --framework=5 --output=model \
    --input_shape="images:1,3,640,640" --soc_version=Ascend910B3

# 带 AIPP 的 OM 转换
atc --model=model.onnx --framework=5 --output=model-aipp \
    --input_shape="images:1,3,640,640" --soc_version=Ascend910B3 \
    --insert_op_conf=aipp.cfg
```

> ATC 在转换时会自动进行**图融合优化**——把相邻小算子合并成大算子，减少 kernel 启动次数与中间结果显存读写，这是 OM 推理比 ONNX Runtime 快数十倍的关键。

---

## 5. 动手实践：在昇腾 NPU 上体验视觉推理

下面我们在昇腾 910B3 NPU 上亲手运行代码，感受智能视觉系统开发的各个环节。

### 5.1 检查运行环境

首先确认 NPU 硬件和 CANN 软件环境是否就绪。

**测试程序说明**：下方代码首先自动安装可能缺失的 Python 依赖（`onnx`、`opencv-python-headless`），并尝试预加载 `libgomp` 动态库以修复 PyTorch 在某些环境下的 TLS 加载问题。然后调用 `npu-smi info` 命令查询 NPU 硬件状态，并检查 CANN 环境变量是否设置。

**预期结果**：如果环境正常，将看到 NPU 设备列表（包含芯片型号 910B3、利用率、显存等信息），以及 `ASCEND_HOME_PATH` 和 `ASCEND_TOOLKIT_HOME` 的路径。如果 `npu-smi` 不可用，说明当前环境没有 NPU 硬件或驱动未安装。依赖安装部分会显示 `[✓]` 表示已安装或安装成功。

In [ ]:
import subprocess, os, sys

# === 环境准备：自动安装缺失依赖 ===
print('=' * 50)
print('环境依赖检查与自动安装')
print('=' * 50)

def _ensure_pkg(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        __import__(import_name)
        print(f'  [✓] {pkg} 已安装')
        return True
    except ImportError:
        print(f'  [!] {pkg} 未安装，正在安装...')
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
            print(f'  [✓] {pkg} 安装完成')
            return True
        except Exception as e:
            print(f'  [✗] {pkg} 安装失败: {e}')
            return False

_ensure_pkg('onnx')
_ensure_pkg('opencv-python-headless', 'cv2')
_ensure_pkg('matplotlib')

# 修复 PyTorch 在某些环境下的 TLS 加载问题（libgomp 无法分配静态 TLS 块）
try:
    import ctypes, glob
    for p in glob.glob('/opt/**/torch.libs/libgomp*.so*', recursive=True):
        try:
            ctypes.CDLL(p, mode=ctypes.RTLD_GLOBAL)
        except Exception:
            pass
except Exception:
    pass

print()
# 检查 NPU 硬件信息
print('=' * 50)
print('昇腾 NPU 环境检查')
print('=' * 50)

try:
    result = subprocess.run(['npu-smi', 'info'], capture_output=True, text=True, timeout=10)
    print(result.stdout)
except Exception as e:
    print(f'npu-smi 不可用: {e}')

# 检查 CANN 环境变量
ascend_home = os.environ.get('ASCEND_HOME_PATH', '未设置')
toolkit_home = os.environ.get('ASCEND_TOOLKIT_HOME', '未设置')
print(f'ASCEND_HOME_PATH: {ascend_home}')
print(f'ASCEND_TOOLKIT_HOME: {toolkit_home}')

### 5.2 用 PyTorch + torch_npu 在 NPU 上做图像处理

模拟视觉系统中的图像预处理与推理计算，在 NPU 上执行。

**测试程序说明**：代码首先检查 `torch.npu.is_available()` 确认 NPU 可用，然后模拟一张 640×640 的 RGB 图像（uint8 随机数据）。将图像数据搬到 NPU 设备（`.to('npu')`）并转为 float32，然后在 NPU 上执行归一化（`/255.0`）——这正是 AIPP 硬件在部署时自动完成的操作。最后创建一个卷积层并放在 NPU 上执行一次推理，验证 NPU 计算能力。

**预期结果**：`NPU 是否可用: True`，`NPU 设备名` 显示昇腾 910B3。模拟图像 shape 为 `(1, 3, 640, 640)`，dtype 为 `uint8`。归一化后数值范围在 `[0.0000, 1.0000]` 之间（因为原始是 0~255 的整数除以 255）。卷积输出 shape 为 `(1, 16, 640, 640)`，设备为 `npu:0`，输出统计的 mean 和 std 是随机值。

**为什么有这样的结果**：归一化将 uint8 的 0~255 映射到 float32 的 0.0~1.0，这是神经网络输入的标准范围。卷积层有 3 输入通道、16 输出通道、3×3 卷积核，输入 `(1,3,640,640)` 经过 padding=1 的卷积后输出尺寸不变，仍为 `(1,16,640,640)`。所有计算在 NPU 的 AI Core 上执行，`output.device` 显示 `npu:0` 证明数据未离开 NPU。

In [ ]:
import torch
import torch_npu
import numpy as np
try:
    import cv2
except ImportError:
    import subprocess, sys; subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'opencv-python-headless', '-q']); import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# === 生成一张丰富的测试图像（模拟街景）===
img = np.zeros((480, 640, 3), dtype=np.uint8)
for y in range(320):  # 天空渐变
    img[y, :] = [int(135 + y * 0.25), int(206 + y * 0.08), int(235)]
img[320:, :] = [34, 139, 34]  # 草地
cv2.circle(img, (520, 80), 45, (0, 215, 255), -1)  # 太阳
img[200:360, 80:240] = (180, 100, 60)  # 房子墙体
for i in range(90):  # 屋顶
    img[200-i:201-i, 80+i:240-i] = (0, 0, 200)
cv2.rectangle(img, (130, 280), (190, 360), (80, 80, 80), -1)  # 门
cv2.rectangle(img, (95, 230), (115, 250), (255, 255, 0), -1)  # 窗
cv2.rectangle(img, (205, 230), (225, 250), (255, 255, 0), -1)
# 画两个人
for (cx, cy, shirt) in [(340, 300, (50, 50, 200)), (410, 310, (200, 50, 50))]:
    cv2.circle(img, (cx, cy - 50), 22, (200, 180, 160), -1)  # 头
    cv2.rectangle(img, (cx - 18, cy - 30), (cx + 18, cy + 60), shirt, -1)  # 身体
cv2.putText(img, 'Vision AI on NPU', (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
os.makedirs('tmp', exist_ok=True)
cv2.imwrite('tmp/demo_scene.jpg', img)
print(f'测试图像已生成: tmp/demo_scene.jpg ({os.path.getsize("tmp/demo_scene.jpg")} bytes)')

# 显示图像
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Generated Scene Image (480x640)'); axes[0].axis('off')

print(f'\nNPU 是否可用: {torch.npu.is_available()}')
if torch.npu.is_available():
    print(f'NPU 设备名: {torch.npu.get_device_name(0)}')

    # 将图像搬到 NPU 并做预处理（这正是 AIPP 硬件做的事）
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (640, 640))
    img_tensor = torch.from_numpy(img_resized).permute(2, 0, 1).unsqueeze(0).to('npu').float()
    print(f'图像 tensor: shape={img_tensor.shape}, dtype={img_tensor.dtype}, device={img_tensor.device}')

    # NPU 归一化 (/255)
    img_norm = img_tensor / 255.0
    print(f'归一化后 range: [{img_norm.min():.4f}, {img_norm.max():.4f}]')

    # NPU 卷积推理
    conv = torch.nn.Conv2d(3, 16, kernel_size=3, padding=1).to('npu')
    with torch.no_grad():
        output = conv(img_norm)
    print(f'卷积输出: shape={output.shape}, device={output.device}')
    print(f'输出统计: mean={output.mean():.4f}, std={output.std():.4f}')

    # 可视化卷积特征图（前 4 个通道）
    feat = output[0, :4].cpu().numpy()
    for i in range(4):
        axes[1].imshow(feat[i], cmap='viridis')
    axes[1].set_title('NPU Conv Feature Map (Channel 0)'); axes[1].axis('off')

    plt.suptitle('NPU Image Processing: Input Image + Conv Feature Map', fontsize=13)
    plt.tight_layout()
    plt.savefig('tmp/npu_features.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'可视化已保存: tmp/npu_features.png')
else:
    axes[1].imshow(cv2.cvtColor(img_resized if 'img_resized' in dir() else img, cv2.COLOR_BGR2RGB))
    axes[1].set_title('NPU 不可用'); axes[1].axis('off')
    plt.tight_layout(); plt.show()
    print('NPU 不可用，请检查环境')

可以看到，图像预处理（归一化）和模型推理（卷积）都在 NPU 上完成。在实际部署中，AIPP 会把归一化固化进 OM 模型，由专用硬件自动完成，无需在 Python 中写 `/255`。

**结果解释**：上述代码中 `img_npu / 255.0` 这一步在 NPU 上执行了逐元素除法，耗时极短。但在实际部署中，如果每次推理都在 Python 中做这个除法，会有两个开销：① CPU 计算 `x/255` 的时间；② 将 float32 结果从 Host 拷贝到 Device 的带宽开销（uint8 是 1 字节/像素，float32 是 4 字节/像素）。AIPP 的作用就是把这一步从 Python 代码中消除，由 NPU 硬件在数据进入网络第一层之前自动完成，既省 CPU 算力又省传输带宽。

### 5.3 对比 CPU 与 NPU 的视觉推理性能

用更大的模型感受 NPU 在视觉推理中的加速效果。

**测试程序说明**：构建一个 `SimpleVisionModel`，包含三层卷积+ReLU+MaxPool 特征提取器和一层全连接分类器，输入为 `(1, 3, 224, 224)`。先在 CPU 上做 3 次热身 + 20 次计时推理，再在 NPU 上做同样的热身和计时（注意 NPU 推理是异步的，需调用 `torch.npu.synchronize()` 等待完成后再计时）。

**预期结果**：CPU 推理约 20~80 ms/次（取决于 CPU 核数和主频），NPU 推理约 0.5~3 ms/次，加速比通常在 10~40 倍之间。

**为什么有这样的结果**：该模型以卷积计算为主，CPU 的通用核心没有矩阵加速单元，只能逐元素做乘加运算；而昇腾 NPU 的 AI Core 内置 Cube 引擎（矩阵乘加速单元），一个时钟周期可完成大量乘加运算。此外，NPU 的片上存储（L1/L2/UB）带宽远高于 CPU 的内存带宽，中间特征图读写也更快。注意 NPU 计时必须先 `synchronize()`，否则测到的是 Python 下发指令的时间而非实际计算时间。

In [ ]:
import time

# 构建一个模拟视觉模型（多层卷积）
class SimpleVisionModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(3, 32, 3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(32, 64, 3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(64, 128, 3, padding=1),
            torch.nn.ReLU(),
            torch.nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = torch.nn.Linear(128, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)

model = SimpleVisionModel()
input_img = torch.randn(1, 3, 224, 224)

# CPU 推理
model_cpu = model.eval()
with torch.no_grad():
    for _ in range(3):  # 热身
        _ = model_cpu(input_img)
    t0 = time.time()
    for _ in range(20):
        _ = model_cpu(input_img)
    cpu_time = (time.time() - t0) / 20 * 1000
print(f'CPU 推理: {cpu_time:.2f} ms/次')

# NPU 推理
if torch.npu.is_available():
    model_npu = model.to('npu').eval()
    input_npu = input_img.to('npu')
    with torch.no_grad():
        for _ in range(3):
            _ = model_npu(input_npu)
        torch.npu.synchronize()
        t0 = time.time()
        for _ in range(20):
            _ = model_npu(input_npu)
        torch.npu.synchronize()
        npu_time = (time.time() - t0) / 20 * 1000
    print(f'NPU 推理: {npu_time:.2f} ms/次')
    print(f'加速比: {cpu_time/npu_time:.1f}x')
else:
    print('NPU 不可用')

# === 性能对比柱状图 ===
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    labels = ['CPU\n(PyTorch)']
    times = [cpu_time]
    colors = ['#4C72B0']
    if 'npu_time' in dir():
        labels.append('NPU\n(Ascend 910B3)')
        times.append(npu_time)
        colors.append('#DD8452')
    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(labels, times, color=colors, width=0.5, edgecolor='black')
    for bar, t in zip(bars, times):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{t:.1f} ms', ha='center', fontsize=12)
    ax.set_ylabel('推理耗时 (ms)', fontsize=12)
    ax.set_title('CPU vs NPU 视觉推理性能对比', fontsize=14)
    if len(times) > 1:
        ax.annotate(f'{times[0]/times[1]:.1f}x 加速', xy=(1, times[1]), xytext=(0.5, max(times)*0.7),
                    fontsize=14, color='red', fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color='red', lw=2))
    plt.tight_layout()
    plt.savefig('tmp/cpu_vs_npu.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('性能对比图已保存: tmp/cpu_vs_npu.png')
except Exception as e:
    print(f'绘图跳过: {e}')

### 5.4 体验模型转换（ATC）

下面演示如何将一个 PyTorch 模型导出为 ONNX，再用 ATC 转换为 OM 模型——这是视觉系统部署的核心步骤。

**测试程序说明**：① 定义一个 `TinyVisionNet`（3→8 通道卷积 + 全连接，输入 32×32），用 `torch.onnx.export` 导出为 ONNX 文件（`opset_version=13`）；② 构造 ATC 命令行，指定 `--framework=5`（ONNX 格式）、`--soc_version=Ascend910B3`（目标芯片）、`--input_shape` 告知输入尺寸，通过 `os.system()` 执行转换。

**预期结果**：ONNX 导出成功，文件约 2~5 KB。ATC 转换成功后生成 `tmp/tiny_vision.om` 文件，大小约 10~50 KB。如果 ATC 返回非零码，通常是因为 CANN 环境未加载（需先 `source set_env.sh`）。

**为什么有这样的结果**：ONNX 文件包含模型的计算图和权重，用 Protobuf 序列化，体积小。ATC 将 ONNX 计算图解析后，针对昇腾 NPU 的硬件特性进行图融合优化（如将 Conv+Bias+ReLU 合并为一个算子），生成 NPU 专用的离线模型指令序列。OM 文件比 ONNX 大是因为包含了编译后的二进制指令和优化的权重布局。

In [ ]:
import os, subprocess, sys

# 确保 onnx 可用（torch.onnx.export 的必需依赖）
try:
    import onnx
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'onnx', '-q'])

# 1. 创建并导出一个简单的 ONNX 模型
os.makedirs('tmp', exist_ok=True)

class TinyVisionNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = torch.nn.Conv2d(3, 8, 3, padding=1)
        self.fc = torch.nn.Linear(8 * 32 * 32, 5)
    def forward(self, x):
        x = torch.relu(self.conv(x))
        x = x.flatten(1)
        return self.fc(x)

model = TinyVisionNet().eval()
dummy = torch.randn(1, 3, 32, 32)
onnx_path = 'tmp/tiny_vision.onnx'
torch.onnx.export(model, dummy, onnx_path, input_names=['images'],
                   output_names=['output'], opset_version=13)
print(f'ONNX 导出成功: {onnx_path} ({os.path.getsize(onnx_path)} bytes)')

# 2. 用 ATC 转换为 OM
om_path = 'tmp/tiny_vision'
atc_cmd = (
    f'atc --model={onnx_path} --framework=5 --output={om_path} '
    f'--input_shape="images:1,3,32,32" --soc_version=Ascend910B3 --log=error'
)
print(f'ATC 命令: {atc_cmd}')
ret = os.system(atc_cmd)
if ret == 0 and os.path.exists(om_path + '.om'):
    print(f'OM 转换成功: {om_path}.om ({os.path.getsize(om_path + ".om")} bytes)')
else:
    print(f'OM 转换返回码: {ret}（可能需要 source CANN 环境变量）')

### 5.5 体验 AIPP 配置与转换

创建 AIPP 配置文件，将预处理固化进 OM 模型。

**测试程序说明**：① 写入 AIPP 配置文件 `tmp/aipp.cfg`，配置静态 AIPP（`aipp_mode: static`），指定输入格式为 `RGB888_U8`（RGB uint8），关闭色域转换（`csc_switch: false`，因 BGR→RGB 已由 Python 完成），设置归一化参数 `min_chn=0.0`、`var_reci_chn=0.00392157`（即 1/255，实现 `y = x/255`）；② 用 ATC 的 `--insert_op_conf=tmp/aipp.cfg` 参数将 AIPP 配置编译进 OM 模型。

**预期结果**：AIPP 配置文件写入成功并打印内容。ATC+AIPP 转换成功后生成 `tmp/tiny_vision_aipp.om`。AIPP-OM 通常比纯 OM 略大（多包含了 AIPP 预处理算子），但推理时输入可以直接传 uint8 数据，无需在 Python 中做归一化。

**为什么有这样的结果**：AIPP 配置中的 `var_reci_chn_0: 0.00392157` 是 1/255 的浮点近似值，NPU 硬件在数据进入网络第一层前自动执行 `y = (x - 0.0) * 0.00392157`，等价于 `y = x / 255`。`csc_switch: false` 是因为 Python 代码中已用 `cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)` 完成了 BGR→RGB 转换，如果 AIPP 再做一次 CSC（Color Space Conversion），就会"同一件事做两遍"导致颜色出错——这是 AIPP 配置最常见的错误。

In [ ]:
# 写入 AIPP 配置文件
aipp_cfg = """aipp_op {
  aipp_mode: static
  related_input_rank: 0
  input_format: RGB888_U8
  src_image_size_w: 32
  src_image_size_h: 32
  crop: false
  load_start_pos_h: 0
  load_start_pos_w: 0
  csc_switch: false
  min_chn_0: 0.0
  min_chn_1: 0.0
  min_chn_2: 0.0
  var_reci_chn_0: 0.00392157
  var_reci_chn_1: 0.00392157
  var_reci_chn_2: 0.00392157
}
"""
with open('tmp/aipp.cfg', 'w') as f:
    f.write(aipp_cfg)
print('AIPP 配置文件已写入: tmp/aipp.cfg')
print(aipp_cfg)

# 用 AIPP 转换 OM
om_aipp_path = 'tmp/tiny_vision_aipp'
atc_aipp_cmd = (
    f'atc --model={onnx_path} --framework=5 --output={om_aipp_path} '
    f'--input_shape="images:1,3,32,32" --soc_version=Ascend910B3 '
    f'--insert_op_conf=tmp/aipp.cfg --log=error'
)
print(f'ATC+AIPP 命令: {atc_aipp_cmd}')
ret = os.system(atc_aipp_cmd)
if ret == 0 and os.path.exists(om_aipp_path + '.om'):
    print(f'AIPP-OM 转换成功: {om_aipp_path}.om')
    # 对比两个 OM 大小
    s1 = os.path.getsize(om_path + '.om') if os.path.exists(om_path + '.om') else 0
    s2 = os.path.getsize(om_aipp_path + '.om')
    print(f'纯 OM: {s1} bytes, AIPP-OM: {s2} bytes')
else:
    print(f'AIPP-OM 转换返回码: {ret}')

## 6. 四条推理路径与性能对比

在昇腾平台上，同一个模型可以通过四条路径推理，性能差异巨大：

<img src="../../images/four_paths.png" alt="四条推理路径" style="display: block; margin-left: 0;" />

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">路径</th><th style="text-align: left;">引擎</th><th style="text-align: left;">预处理位置</th><th style="text-align: left;">参考耗时</th><th style="text-align: left;">加速来源</th></tr>
<tr><td style="text-align: left;">① PyTorch</td><td style="text-align: left;">torch (CPU)</td><td style="text-align: left;">Python/CPU</td><td style="text-align: left;">83+ ms</td><td style="text-align: left;">基准</td></tr>
<tr><td style="text-align: left;">② ONNX Runtime</td><td style="text-align: left;">onnxruntime (CPU)</td><td style="text-align: left;">Python/CPU</td><td style="text-align: left;">~83 ms</td><td style="text-align: left;">图优化</td></tr>
<tr><td style="text-align: left;">③ 纯 OM</td><td style="text-align: left;">AscendCL (NPU)</td><td style="text-align: left;">Python/CPU</td><td style="text-align: left;">~2.8 ms</td><td style="text-align: left;">NPU 算力 + 图融合</td></tr>
<tr><td style="text-align: left;">④ AIPP-OM</td><td style="text-align: left;">AscendCL (NPU)</td><td style="text-align: left;">NPU 硬件</td><td style="text-align: left;">~1.35 ms</td><td style="text-align: left;">+ AIPP 预处理卸载</td></tr>
</table>

上表是本课程最重要的性能对比数据。路径①PyTorch 在 CPU 上推理，是基准线，耗时约 83 ms。路径②ONNX Runtime 同样在 CPU 上推理，但通过图优化（算子融合、常量折叠）略微提速，耗时约 83 ms，与 PyTorch 接近，说明 CPU 上图优化的收益有限。路径③纯 OM 将模型编译为昇腾 OM 格式后在 NPU 上推理，耗时降至约 2.8 ms，相比 CPU 路径加速约 30 倍，加速来源是 NPU 的专用 AI 算力（矩阵单元、Cube 引擎）和 ATC 的图融合优化。路径④AIPP-OM 在纯 OM 基础上把预处理也卸载到 NPU 硬件，耗时进一步降至约 1.35 ms，相比纯 OM 再加速约 2 倍，加速来源是 AIPP 的计算卸载（归一化在 NPU 硬件完成）、带宽节省（uint8 替代 float32，传输量降为 1/4）和流水衔接（预处理与推理在 NPU 内部无缝衔接）。

从 CPU 到 NPU+AIPP，端到端加速约 **60 倍**！这正是昇腾视觉系统高性能的核心所在。

---

## 小结

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">概念</th><th style="text-align: left;">一句话理解</th></tr>
<tr><td style="text-align: left;">智能视觉系统</td><td style="text-align: left;">让计算机看懂图像/视频并做决策的系统</td></tr>
<tr><td style="text-align: left;">DVPP</td><td style="text-align: left;">NPU 内置硬件视频预处理单元（解码/缩放/色彩转换）</td></tr>
<tr><td style="text-align: left;">AIPP</td><td style="text-align: left;">固化进 OM 模型的硬件 AI 预处理（归一化/通道转换）</td></tr>
<tr><td style="text-align: left;">AscendCL</td><td style="text-align: left;">管理 NPU 资源、加载模型、执行推理的 API</td></tr>
<tr><td style="text-align: left;">ATC</td><td style="text-align: left;">将 ONNX 转为 NPU 专属 OM 的模型转换工具</td></tr>
<tr><td style="text-align: left;">部署链路</td><td style="text-align: left;">PT → ONNX → OM → OM+AIPP</td></tr>
</table>

本节建立了智能视觉系统开发的完整认知框架。核心要点回顾：**智能视觉系统**是让计算机看懂图像并做决策的系统，由数据层、硬件层、CANN 层、框架层、应用层五层构成。**DVPP** 是 NPU 内置的硬件视频预处理单元，用专用硬件加速 JPEG 解码和图像缩放，与 AI Core 共享芯片，无需跨设备数据搬运。**AIPP** 是固化进 OM 模型的硬件 AI 预处理，把归一化和通道转换从 CPU 卸载到 NPU，输入数据从 float32 降为 uint8，带宽节省 3/4。**AscendCL** 是管理 NPU 资源和执行推理的 C++/Python API。**ATC** 将 ONNX 编译为昇腾专属 OM 模型，期间自动进行图融合优化。部署链路 `PT → ONNX → OM → OM+AIPP` 每一步都带来性能提升，从 CPU 的 83 ms 到 NPU+AIPP 的 1.35 ms，端到端加速约 60 倍。

---

## 课后练习

请根据本节课程学习内容完成以下题目进行自测。

**第1题**（单选题）智能视觉系统开发流程中，将训练好的模型部署到昇腾 NPU 上需要经过哪一步转换？

- A. 直接使用 .pt 文件推理
- B. 将 .pt/.onnx 转换为 .om 格式
- C. 将模型编译为 CUDA kernel
- D. 不需要转换，NPU 兼容所有格式


In [ ]:
q1 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第{1}题答案已记录：{q1}' if q1 else '⚠️ 请填入答案并运行本单元格')

**第2题**（单选题）DVPP 的主要功能是什么？

- A. 模型训练加速
- B. 硬件视频预处理（JPEG解码、缩放、色彩转换）
- C. 算子编译
- D. 分布式通信


In [ ]:
q2 = ''  # 填入你的选项，如 'B'
print(f'第{2}题答案已记录：{q2}' if q2 else '⚠️ 请填入答案并运行本单元格')

**第3题**（单选题）AIPP 把预处理固化进 OM 模型后，输入数据类型从 float32 变为 uint8，带宽节省了多少？

- A. 1/2
- B. 1/3
- C. 1/4（即原来的 1/4）
- D. 没有节省


In [ ]:
q3 = ''  # 填入你的选项，如 'C'
print(f'第{3}题答案已记录：{q3}' if q3 else '⚠️ 请填入答案并运行本单元格')

**第4题**（单选题）ATC 转换模型时，通过哪个参数指定 AIPP 配置文件？

- A. --aipp_config
- B. --insert_op_conf
- C. --preprocess
- D. --input_format


In [ ]:
q4 = ''  # 填入你的选项，如 'B'
print(f'第{4}题答案已记录：{q4}' if q4 else '⚠️ 请填入答案并运行本单元格')

**第5题**（单选题）以下哪种推理路径性能最高？

- A. PyTorch CPU 推理
- B. ONNX Runtime CPU 推理
- C. 纯 OM NPU 推理
- D. AIPP-OM NPU 推理（硬件预处理）


In [ ]:
q5 = ''  # 填入你的选项，如 'D'
print(f'第{5}题答案已记录：{q5}' if q5 else '⚠️ 请填入答案并运行本单元格')

**第6题**（单选题）AscendCL 中加载 OM 模型的 API 是？

- A. acl.rt.set_device
- B. acl.mdl.load_from_file
- C. acl.rt.malloc
- D. acl.mdl.execute


In [ ]:
q6 = ''  # 填入你的选项，如 'B'
print(f'第{6}题答案已记录：{q6}' if q6 else '⚠️ 请填入答案并运行本单元格')

**第7题**（单选题）AIPP 归一化公式 `y = (x - min_chn) * var_reci_chn` 中，要实现 `y = x/255`，`var_reci_chn` 应设为？

- A. 255.0
- B. 0.00392157（即 1/255）
- C. 1.0
- D. 0.5


In [ ]:
q7 = ''  # 填入你的选项，如 'B'
print(f'第{7}题答案已记录：{q7}' if q7 else '⚠️ 请填入答案并运行本单元格')

**第8题**（单选题）OM 推理比 ONNX Runtime 推理快数十倍的主要原因是？

- A. OM 文件更小
- B. ATC 的图融合优化减少了 kernel 启动和显存读写
- C. OM 使用了更高精度
- D. ONNX Runtime 不支持推理


In [ ]:
q8 = ''  # 填入你的选项，如 'B'
print(f'第{8}题答案已记录：{q8}' if q8 else '⚠️ 请填入答案并运行本单元格')

**全部作答完成后，运行下方代码查看批改结果：**


In [ ]:
import sys
from pathlib import Path

for candidate in (
    Path.cwd() / 'answer',
    Path.cwd() / '06_vision_dev' / 'answer',
):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find answer directory')
from grade_05 import grade
grade(globals())

## 参考资料

- [昇腾 DVPP 文档](https://www.hiascend.com/document)
- [昇腾 AIPP 配置指南](https://www.hiascend.com/document/detail/zh/CANNCommunityEdition)
- [AscendCL API 参考](https://www.hiascend.com/document/detail/zh/CANNCommunityEdition)
- [ATC 模型转换工具](https://www.hiascend.com/document/detail/zh/CANNCommunityEdition)
- [实验6.1：开发板 DVPP 与 AIPP 的 YOLO 目标检测](./exp6_ascend_orangepi_local/README.md)
- [实验6.2：云平台的 AIPP 测试](./exp6_cann_sandbox/README.md)